# Question → JSON plan translator: training on Colab

Fine-tunes a small model (LoRA) to turn a short question into a JSON plan, using `train.py` and
generated training data.

1. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
2. Nothing is saved to Google Drive unless you set `USE_DRIVE = True` in the first code cell. With it, a dropped
   session continues from the last checkpoint when you **Run all** again; without it, training starts over.
3. At the end the plans for the test and probe questions download (`*.predictions.jsonl`), then the model. The app scores the plans.

In [ ]:
# Settings
MODEL = "Qwen/Qwen3-0.6B"      # google/gemma-3-270m-it needs a Hugging Face login and its licence accepted
RUN_NAME = "qwen06-v3"    # change this to start a fresh run instead of continuing this one
BATCH = 8
MAX_STEPS = -1                # -1: one full pass over the training rows (about 2 h on a T4); a number stops earlier
SPEED_CHECK = False           # True runs 30 short steps first and estimates the full run
USE_DRIVE = False              # True: checkpoints go to Google Drive, so a dropped session continues instead of starting over

import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = f"/content/drive/MyDrive/plan-translator/{RUN_NAME}"
else:
    OUT = f"/content/runs/{RUN_NAME}"
os.makedirs(os.path.dirname(OUT), exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# The library versions train.py was tested with
!pip install -q transformers==5.22.0 trl==1.17.0 peft==0.21.2 datasets==5.1.0 accelerate==1.15.0
# Colab ships an old torchao that peft refuses to load next to; nothing here uses it
!pip uninstall -q -y torchao


In [ ]:
# Scripts and data
!rm -rf /content/repo && git clone -q --depth 1 https://github.com/fatihsokmen/plan-translator-colab /content/repo
!tar xzf /content/repo/data.tar.gz -C /content/repo && wc -l /content/repo/data/*/*.jsonl

In [ ]:
# Speed check: 30 short steps, then an estimate for the full run (this Mac: about 7 s per step)
if SPEED_CHECK:
    import math, re
    !cd /content/repo && python train.py --model {MODEL} --data data --out /content/speedtest --limit 400 --max_steps 30 --bs {BATCH} --dev_rows 5 --eval_rows 20 > /content/speedtest.log 2>&1
    log = open("/content/speedtest.log").read()
    m = re.search(r"'train_runtime': '?([\d.]+)", log)
    if m:
        per_step = float(m.group(1)) / 30
        steps = math.ceil(sum(1 for _ in open("/content/repo/data/train/train.jsonl")) / BATCH)
        print(f"{per_step:.2f} s per step → full run about {steps * per_step / 3600:.1f} h ({steps} steps)")
    else:
        print(log[-3000:])
        raise RuntimeError("speed check failed: the error is in the log above")

In [ ]:
# Training: MAX_STEPS steps over the training rows. Prints the dev exact-plan rate at the end.
!cd /content/repo && python train.py --model {MODEL} --data data --out {OUT} --bs {BATCH} --save_steps 200 --max_steps {MAX_STEPS} 2>&1 | tee -a {OUT}.log
assert os.path.exists(f"{OUT}/final"), "training did not finish: the error is in the output above"

In [ ]:
# The first model's dev questions (300, none of them in this run's training data): exact-plan rate to compare with 0.870.
# "biggest ..." questions without a count got a random count in the first data and none now, so they're also scored without it.
import json
!cd /content/repo && python predict.py --model {OUT}/final --prompts data/dev_v1_300.prompts.jsonl --out {OUT}/dev_v1_300.predictions.jsonl
rows = {r["id"]: r for r in map(json.loads, open("/content/repo/data/dev_v1_300.gold.jsonl"))}
def plan(t):
    try: return json.loads(t[t.index("{"): t.rindex("}") + 1])
    except ValueError: return None
def uncounted(r):
    g = json.loads(r["output"])
    return g.get("intent") == "list" and g.get("order") == "largest" and not any(ch.isdigit() for ch in r["input"].split("\n")[0])
pred = {json.loads(l)["id"]: plan(json.loads(l)["output"]) for l in open(f"{OUT}/dev_v1_300.predictions.jsonl")}
exact = sum(pred[i] == json.loads(r["output"]) for i, r in rows.items())
loose = sum((pred[i] or {}) | {"limit": 0} == json.loads(r["output"]) | {"limit": 0} if uncounted(r) else pred[i] == json.loads(r["output"]) for i, r in rows.items())
print(f"old dev: exact plan {exact / len(rows):.3f} ({exact}/{len(rows)}); ignoring the count on uncounted 'biggest' questions {loose / len(rows):.3f}; first model: 0.870 (261/300)")

In [ ]:
# Plans for the frozen test set, the independent test set and the combination probe set; the app scores these files
from google.colab import files
for name, prompts in [("real_like_v2", "data/test/real_like_v2.prompts.jsonl"), ("independent_v3", "data/test/independent_v3.prompts.jsonl"), ("probe_combos_v1", "data/probe/combos_v1.prompts.jsonl")]:
    !cd /content/repo && python predict.py --model {OUT}/final --prompts {prompts} --out {OUT}/{name}.predictions.jsonl
    files.download(f"{OUT}/{name}.predictions.jsonl")
files.download(f"{OUT}/dev_v1_300.predictions.jsonl")

In [ ]:
# The model itself, as bf16 (about 1.2 GB): Colab deletes it when the session ends
import torch, shutil
from transformers import AutoModelForCausalLM, AutoTokenizer
m = AutoModelForCausalLM.from_pretrained(f"{OUT}/final", dtype=torch.bfloat16)
m.save_pretrained("/content/model_bf16")
AutoTokenizer.from_pretrained(f"{OUT}/final").save_pretrained("/content/model_bf16")
shutil.make_archive(f"/content/{RUN_NAME}", "zip", "/content/model_bf16")
files.download(f"/content/{RUN_NAME}.zip")